# Supervised Fine-Tuning of GaMS-1B for Sentiment Classification

In this notebook, we will fine-tune [GaMS-1B](https://huggingface.co/cjvt/GaMS-1B) - a 1-billion-parameter Slovenian causal language model — for a three-class sentiment classification task using **Supervised Fine-Tuning (SFT)**. Unlike BERT-style encoder models that output a class label directly, a decoder-type model like GaMS is trained to *generate* the correct label as text.

Because GaMS-1B is too large to fine-tune all its parameters on typical hardware, we use **LoRA (Low-Rank Adaptation)**, a parameter-efficient fine-tuning (PEFT) technique that trains only a small number of added adapter weights while keeping the base model frozen. This drastically reduces memory consumption and training time while preserving most of the model's pretrained knowledge.

The notebook covers the full SFT pipeline: loading the dataset, formatting examples as instruction-following prompt/completion pairs, configuring LoRA and the SFT trainer, training, and merging the adapter weights back into the base model for inference.

In [7]:
!pip3 install trl transformers accelerate datasets
!pip3 install --upgrade torchao


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Load Data

We will work with the dataset prepared in the previous task. The processed version is already available on HuggingFace and can be downloaded directly with the `load_dataset` function. We extract both the training split and the validation split — the latter is used as a held-out set during training to monitor performance and guard against overfitting.

In [3]:
from datasets import load_dataset

dataset = load_dataset("zivast/KKS-sentiment-workshop")
train_dataset = dataset["train"]
print("Train data size", len(train_dataset))
val_dataset = dataset["validation"]
print("Val data size", len(val_dataset))

train_dataset

/home/nikola/projects/studies/slaif/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train data size 3821
Val data size 478


Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 3821
})

## Load Model

We load [GaMS-1B](https://huggingface.co/cjvt/GaMS-1B), a 1-billion-parameter autoregressive language model trained on Slovenian text. As a **decoder-only (causal LM)** model, it is loaded with `AutoModelForCausalLM` — unlike BERT which uses `AutoModelForSequenceClassification`. The model does not have a classification head; instead, it will be trained to *generate* the correct sentiment label as a text token.

We use two separate HuggingFace repositories: the base model weights (`cjvt/GaMS-1B`) and the tokenizer from the instruction-tuned chat variant (`cjvt/GaMS-1B-Chat`). We chose the base model instead of the instruction tuned as we are training it for a specific task, that the model did not observe during the instruction tuning. However, we need to use the chat tokenizer as it includes the chat template needed to correctly format prompt/completion pairs. We use `device_map="auto"` to let the `transformers` library automatically distribute the model across available GPU(s).

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "cjvt/GaMS-1B"
tokenizer_model = "cjvt/GaMS-1B-Chat"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)

W0611 13:45:14.132000 1158127 .venv/lib/python3.14/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0611 13:45:14.148000 1158127 .venv/lib/python3.14/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 720.71it/s]


## Prepare data

Data preparation for a decoder-type model is more involved than for BERT. Instead of simply tokenizing the input text, we need to format each example as an **instruction-following prompt/completion pair** that the model can learn from.

We define a Slovenian instruction (`PROMPT_INSTRUCTION`) that tells the model what task to perform: classify the sentiment of the given text into one of three classes — *pozitiven* (positive), *nevtralen* (neutral), or *negativen* (negative). The input text is wrapped in a `<besedilo>` tag and the expected output in a `<sentiment>` tag. These structural markers make it easy for the model to locate relevant parts of the input and produce a well-structured output.

The `example_to_prompt` function converts each dataset example into a dictionary with two fields:
- **`prompt`**: the user turn containing the instruction and input text, formatted as a chat message.
- **`completion`**: the assistant turn containing the expected sentiment label.

We also convert the English label names (`positive`, `negative`, `neutral`) to their Slovenian equivalents to stay consistent with the instruction language. The `map` function applies this transformation to every example in the dataset using 8 parallel workers for speed.

After mapping, we apply the model's chat template to the prompt to inspect how the final formatted input will look. Note that during training we will use `completion_only_loss=True`, which means the loss is computed **only on the completion tokens** — the model is not penalized for "predicting" the instruction prompt it already received as input.

In [8]:
PROMPT_INSTRUCTION = """\
Določi sentiment podanega besedila. Sentiment razvrsti v enega od naslednjih 3 razredov: 'pozitiven', 'nevtralen', 'negativen'

Vhodno besedilo bo podano znotraj oznake '<besedilo>'. Sentiment vrni znotraj oznake '<sentiment>'. Ne vračaj ničesar drugega.
"""

def label_to_slovene(label):
    if label == "positive":
        return "pozitiven"
    if label == "negative":
        return "negativen"
    if label == "neutral":
        return "nevtralen"
    
    raise ValueError("Invalid label:", label)

def example_to_prompt(example):
    input_text = example["text"].strip()
    prompt = f"{PROMPT_INSTRUCTION}\n\n<besedilo>\n{input_text}\n</besedilo>"

    slovene_label = label_to_slovene(example["label_text"])
    completion = f"<sentiment>\n{slovene_label}\n</sentiment>"

    return {
        "prompt": [
            {"role": "user", "content": prompt}
        ],
        "completion": [
            {"role": "assistant", "content": completion}
        ]
    }

In [9]:
from pprint import pprint

train_dataset = train_dataset.map(example_to_prompt, num_proc=8)
val_dataset = val_dataset.map(example_to_prompt, num_proc=8)

first_example = train_dataset[0]
print("First training example:")
pprint(first_example, sort_dicts=False)
print("\nPrompt:")
print(tokenizer.apply_chat_template(first_example["prompt"], tokenize=False))


First training example:
{'text': 'Samo 30% igralcev lahko preseže mejo 100 metrov na smučarski '
         'skakalnici. Vašemu prijatelju je že uspelo. Lahko tudi vam? Pojdite '
         'skakat na: s1.skijumpmania.com/r50261',
 'label': 1,
 'label_text': 'neutral',
 'prompt': [{'role': 'user',
             'content': 'Določi sentiment podanega besedila. Sentiment '
                        'razvrsti v enega od naslednjih 3 razredov: '
                        "'pozitiven', 'nevtralen', 'negativen'\n"
                        '\n'
                        'Vhodno besedilo bo podano znotraj oznake '
                        "'<besedilo>'. Sentiment vrni znotraj oznake "
                        "'<sentiment>'. Ne vračaj ničesar drugega.\n"
                        '\n'
                        '\n'
                        '<besedilo>\n'
                        'Samo 30% igralcev lahko preseže mejo 100 metrov na '
                        'smučarski skakalnici. Vašemu prijatelju je že uspelo. '
  

## Setup Training

Setting up training for GaMS-1B is more complex than for BERT due to the model's size and the use of LoRA. The configuration involves two objects: a **LoRA configuration** and an **SFT training configuration**.

### LoRA Configuration

We use **LoRA (Low-Rank Adaptation)** from the `peft` library to reduce the number of trainable parameters. Instead of updating all ~1 billion base model weights, LoRA injects small trainable low-rank matrices into the attention layers of the transformer. Key parameters:
- `r=16`: the rank of the low-rank matrices — higher rank means more expressive adapters but more trainable parameters.
- `lora_alpha=32`: scaling factor for the LoRA weight updates, effectively controlling the magnitude of the adapter's contribution.
- `lora_dropout=0.05`: dropout applied to the LoRA layers for regularization.
- `bias="none"`: bias parameters are not trained.

### Filtering Long Examples

Before configuring the trainer, we filter out examples whose **prompt alone** already exceeds `MAX_LENGTH=512` tokens. We again limit the total length to 512 tokens, but this time not due to the context window (GaMS-1B has context window of 2048 tokens) but to limit the memory usage during the training. Unlike BERT where we truncated long inputs, here we discard them. The reason for this is that inputs, that are longer than maximum allowed length are truncated during the `SFTTrainer` data preparation. If we combine that with completion only loss, we could get examples where all the tokens would have loss masked. Moreover, truncating the prompt mid-sentence would corrupt the instruction structure and potentially confuse the model. Removing these examples is the safer approach.

### SFT Training Arguments

We use `SFTConfig` from the `trl` library (an extension of `TrainingArguments`) with the following notable settings:
- `completion_only_loss=True`: computes cross-entropy loss **only on the completion tokens** (the assistant's response). This is the standard SFT training objective — the model learns to generate the correct label, not to memorize the prompt.
- `micro_batch_size=1`: the model is too large to fit more than one example per GPU at a time; gradient accumulation over 16 steps achieves an effective global batch size of 16.
- `gradient_checkpointing=True`: trades additional compute for reduced GPU memory by recomputing activations during the backward pass instead of storing them.
- `learning_rate=5e-5`
- Cosine learning rate schedule with 100 linear warmup steps and a minimum LR of `5e-6`.
- Evaluation and checkpointing are performed twice per epoch.

If you want to know more about the available hyperparameters, check the [SFTConfig documentation](https://huggingface.co/docs/trl/en/sft_trainer#trl.SFTConfig).

In [10]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [11]:
MAX_LENGTH = 512

def remove_long_examples(tokenizer, max_length):
    """
    Removes examples from HF dataset, where prompt exceeds max_length.
    """

    def filter_fn(example):
        prompt_tokens = tokenizer.apply_chat_template(example["prompt"], tokenize=True, add_generation_prompt=True, return_dict=False)
        return len(prompt_tokens) < max_length

    return filter_fn

train_dataset = train_dataset.filter(remove_long_examples(tokenizer, MAX_LENGTH), num_proc=8)
val_dataset = val_dataset.filter(remove_long_examples(tokenizer, MAX_LENGTH), num_proc=8)

print("Remaining train examples:", len(train_dataset))
print("Remaining val examples:", len(val_dataset))


Remaining train examples: 3775
Remaining val examples: 473


In [12]:
from trl import SFTConfig, SFTTrainer

# Batch size per GPU
micro_batch_size = 1
# Global batch size
batch_size = 16
# Accumulate the gradients to achieve the batch size
gradient_accumulation_steps = batch_size // micro_batch_size

# Number of training epochs
num_epochs = 2
steps_per_epoch = len(train_dataset) // batch_size
eval_steps = int(1 / 2 * steps_per_epoch)  # Evaluate 2 times per epoch
save_steps = int(1 / 2 * steps_per_epoch)  # Save 2 times per epoch

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Evaluate each {eval_steps} steps ({eval_steps / steps_per_epoch:.2f} epochs)")
print(f"Save each {save_steps} steps ({save_steps / steps_per_epoch:.2f} epochs)")

args = SFTConfig(
    completion_only_loss=True,
    max_length=MAX_LENGTH,
    # Output settings
    output_dir="tmp",  # Directory to save model checkpoints
    # Training duration
    num_train_epochs=num_epochs,
    # Batch size settings
    per_device_train_batch_size=micro_batch_size,
    per_device_eval_batch_size=micro_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    # Memory optimization
    gradient_checkpointing=True,  # Trade compute for memory savings
    # Optimizer settings
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=5e-5,  # Learning rate (QLoRA paper)
    # Learning rate schedule
    warmup_steps=100,  # Number of steps when LR is linearly increasing
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": 5e-6},
    # Logging and saving
    logging_steps=10,  # Log metrics every N steps
    eval_on_start=True, # Perform first evaluation before starting the training
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    # Precision settings
    bf16=True,
    # Integration settings
    push_to_hub=False,  # Don't push to HuggingFace Hub
    report_to="none"  # Disable external logging
)

Steps per epoch: 235
Evaluate each 117 steps (0.50 epochs)
Save each 117 steps (0.50 epochs)


## Train

We initialize `SFTTrainer` from the `trl` library, which extends the standard `Trainer` with SFT-specific functionality such as prompt/completion formatting and the `completion_only_loss` objective. It also integrates seamlessly with `peft` - passing the `peft_config` is enough for the trainer to automatically wrap the base model in LoRA adapters before training begins.

We pass the model, tokenizer, training and validation datasets, `SFTConfig` arguments, and the `LoraConfig`. The trainer handles tokenization with the chat template, data collation, mixed-precision training, gradient accumulation, and periodic evaluation internally.

In [13]:
trainer = SFTTrainer(
    model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=args,
    peft_config=peft_config
)

trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
0,No log,1.553665,2.199581,0.000000,0.723991
117,0.060882,0.046962,0.055448,296181.000000,0.982563
234,0.045022,0.040385,0.039313,590692.000000,0.984587
351,0.034582,0.034970,0.026335,883579.000000,0.986852
468,0.041624,0.034349,0.027998,1180627.000000,0.987637
472,0.043764,0.034293,0.027952,1190044.000000,0.987627


TrainOutput(global_step=472, training_loss=0.1723415523104496, metrics={'train_runtime': 614.9697, 'train_samples_per_second': 12.277, 'train_steps_per_second': 0.768, 'total_flos': 9822063988604928.0, 'train_loss': 0.1723415523104496, 'epoch': 2.0})

## Save model

Saving a LoRA fine-tuned model requires an extra step compared to a full fine-tuning workflow. After training, only the **adapter weights** are stored in the checkpoint directory - not the full model. To produce a standalone model that can be used for inference without the `peft` library, we need to **merge** the LoRA adapters back into the base model weights.

We first load the best checkpoint using `PeftModel.from_pretrained`, which loads the base model and attaches the saved adapter weights on top. We then call `merge_and_unload()`, which mathematically folds the LoRA matrices into the corresponding base model weight matrices and returns a standard `AutoModelForCausalLM` with no adapter overhead. The merged model and tokenizer are then saved normally with `save_pretrained`.

The resulting `GaMS-1B-Sent` directory contains a fully self-contained model that can be loaded and used for inference with any standard `transformers` pipeline - identical in structure to the original base model.

In [14]:
from peft import PeftModel

adapter_checkpoint = "tmp/checkpoint-472"
model = PeftModel.from_pretrained(model, adapter_checkpoint)

model_output_path = "GaMS-1B-Sent"
model = model.merge_and_unload()
model.save_pretrained(model_output_path)
tokenizer.save_pretrained(model_output_path)

/home/nikola/projects/studies/slaif/.venv/lib/python3.14/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.84s/it]


('GaMS-1B-Sent/tokenizer_config.json',
 'GaMS-1B-Sent/chat_template.jinja',
 'GaMS-1B-Sent/tokenizer.json')